# Species Abundance Modeling: Wind Energy and Conservation Reserve Program Effects

**Authors:** [Author Names]  
**Journal:** [Journal Name]  
**DOI:** [Article DOI]  

---

This notebook contains the full analytical pipeline used in the associated research article. It is structured to mirror the Methods section of the paper and walks through:

1. eBird data extraction and preparation
2. Spatial and temporal filtering and subsampling
3. Joining land cover, wind turbine, and CRP datasets
4. Variable categorization and correlation analysis
5. Standardization and train/test splitting
6. Distribution family evaluation (null models)
7. Univariate variable screening (effort, land cover, wind, CRP)
8. Full candidate model fitting (wind-only, CRP-only, combined)
9. Model diagnostics
10. Figure generation

> **Note:** All species names, file paths, and spatial grain parameters have been generalized. Replace placeholder values (e.g., `species_name_here`, `wind_variable`, `crp_variable`) with your specific inputs before running.

---
## Part 1: eBird Data Extraction and Preparation

Raw eBird Basic Dataset (EBD) files are filtered using the `auk` package. Observations are zero-filled to include non-detection checklists, then cleaned and filtered by effort constraints.

In [ ]:
# Libraries
# Data manipulation & tidy tools
library(tidyverse)
library(lubridate)
# Spatial & raster data
library(sf)
library(raster)
library(dggridR)
library(rnaturalearth)
# Plotting & graphics
library(ggpubr)
library(gridExtra)
library(viridis)
library(fields)
library(corrplot)
# Statistical modeling
library(mgcv)
library(pscl)
library(MASS)
library(FSA)
library(fitdistrplus)
library(hglm)
library(zigam)
library(grplasso)
# Model comparison & diagnostics
library(AICcmodavg)
library(MuMIn)
library(DescTools)
library(pdp)
# eBird specific
library(auk)
library(ebirdst)

# Resolve namespace conflicts
select <- dplyr::select

# Set working directory
setwd("path/to/your/project/data")

In [ ]:
# Setup auk eBird data (update paths as appropriate)
ebd <- auk_ebd("raw_data/ebd_species_raw.txt", file_sampling = "raw_data/ebd_sampling_raw.txt")

# Initial filters (replace species name with your target species)
ebd_filters <- ebd %>%
  auk_species("species_name_here") %>%
  auk_year(year = 2012:2021) %>%
  auk_date(date = c("*-06-14", "*-08-03")) %>%
  auk_protocol(protocol = c("Stationary", "Traveling")) %>%
  auk_complete()

# Output directory and paths
data_dir <- "data"
if (!dir.exists(data_dir)) dir.create(data_dir)

f_ebd      <- file.path(data_dir, "ebd_species_breed.txt")
f_sampling <- file.path(data_dir, "ebd_sampling_species_breed.txt")

# Filter and write if not already done
if (!file.exists(f_ebd)) {
  auk_filter(ebd_filters, file = f_ebd, file_sampling = f_sampling)
}

In [ ]:
# Zero-fill to include non-detections
ebd_zf <- auk_zerofill(f_ebd, f_sampling, collapse = TRUE)

# Helper: convert HH:MM:SS to decimal hours
time_to_decimal <- function(x) {
  x <- hms(x, quiet = TRUE)
  hour(x) + minute(x) / 60 + second(x) / 3600
}

# Clean and transform variables
ebd_zf <- ebd_zf %>%
  mutate(
    observation_count = if_else(observation_count == "X", NA_character_, observation_count),
    observation_count = as.integer(observation_count),
    effort_distance_km = if_else(protocol_type != "Traveling", 0, effort_distance_km),
    time_observations_started = time_to_decimal(time_observations_started),
    year = year(observation_date),
    day_of_year = yday(observation_date)
  )

# Filter by effort constraints
ebd_filtered <- ebd_zf %>%
  filter(
    duration_minutes  <= 300,
    effort_distance_km <= 5,
    number_observers  <= 10
  )

# Select relevant columns
ebird_data <- ebd_filtered %>%
  select(
    checklist_id, observer_id, sampling_event_identifier,
    scientific_name, observation_count, species_observed,
    state_code, locality_id, latitude, longitude,
    protocol_type, all_species_reported,
    observation_date, year, day_of_year,
    time_observations_started,
    duration_minutes, effort_distance_km, number_observers
  )

---
## Part 2: Spatial Filtering, Hexagonal Grid Assignment, and Subsampling

Checklists are clipped to the species breeding range within the US border. A discrete global grid (dggridR) assigns each checklist to a spatial cell. Data are then summarized per cell-year and subsampled to equalize annual effort.

In [ ]:
# Load spatial boundaries
USborder          <- st_read("shapefiles/US_border.shp") %>% st_transform(crs = 4326)
rangemap_allseasons <- st_read("range_maps/species_range.gpkg")

# Subset to breeding season and clip to US
rangemap_season <- filter(rangemap_allseasons, season == "breeding")
rangemap        <- st_intersection(rangemap_season, USborder)

# Convert checklists to sf points and intersect with range
ebird_sf       <- st_as_sf(ebird_data, coords = c("longitude", "latitude"), crs = 4326)
ebird_filtered <- st_intersection(ebird_sf, rangemap) %>% as_tibble()

# Extract lat/lon from geometry
ebird_filtered <- ebird_filtered %>%
  mutate(
    longitude = map_dbl(geometry, 1),
    latitude  = map_dbl(geometry, 2)
  )

# Remove checklists with missing counts
ebird_filtered <- ebird_filtered %>% filter(!is.na(observation_count))

In [ ]:
# Construct hexagonal grid (~4.5 km spacing) and assign cells
dggs <- dgconstruct(spacing = 9)

ebird_filtered <- ebird_filtered %>%
  mutate(
    cell = dgGEO_to_SEQNUM(dggs, longitude, latitude)$seqnum,
    week = week(observation_date)
  )

# Summarize: mean abundance and effort per cell-year
ebird_summary <- ebird_filtered %>%
  mutate(cell_year = paste(cell, year, sep = "_")) %>%
  group_by(cell_year) %>%
  summarise(
    meanCount     = mean(observation_count),
    year          = max(year),
    meanStartTime = mean(time_observations_started),
    meanSampDur   = mean(duration_minutes),
    meanDIST      = mean(effort_distance_km),
    meanNumObs    = mean(number_observers)
  ) %>%
  ungroup() %>%
  mutate(species_observed = if_else(meanCount > 0, "TRUE", "FALSE"))

# Subsample to equalize effort across years
min_year_count <- min(table(ebird_summary$year))
ebird_summary_ss <- ebird_summary %>%
  group_by(year) %>%
  sample_n(size = min_year_count) %>%
  ungroup()

# Save cleaned data
write_csv(ebird_summary_ss, file.path(data_dir, "species_breed_finaldata.csv"))

---
## Part 3: Joining Ancillary Datasets

The core eBird summary is joined with land cover, wind turbine, and Conservation Reserve Program (CRP) datasets. Observations are classified as experimental (presence of wind turbines or CRP) or control (neither). Control points are subsampled to reduce imbalance in developed land cover relative to experimental points.

In [ ]:
# Load bird summary and keep relevant columns
spec_raw_spatialgrain   <- read_csv(file.choose())  # species_season_finaldata.csv
spec_clean_spatialgrain <- spec_raw_spatialgrain[, c(1:11, 36:45)]

# Join land cover (proportion of landscape by class)
landcover <- read.csv("path/to/landcover4.5_cell_PLAND.csv", header = TRUE)
landcover <- landcover %>% select(-seqnum, -year)
spec_joined_spatialgrain <- left_join(spec_clean_spatialgrain, landcover, by = "cell_year")

# Join wind turbine data (fill NAs with 0 = no turbines)
turbines <- read.csv(file.choose(), header = TRUE)  # WRD_foranalysis.csv
spec_joined_spatialgrain <- left_join(spec_joined_spatialgrain, turbines, by = "cell_year") %>%
  mutate(WindCount = ifelse(is.na(WindCount), 0, WindCount))

# Join CRP area by seqnum + year
CRP <- read.csv(file.choose(), header = TRUE)
CRP$year <- CRP$Year
spec_joined_spatialgrain <- left_join(
  spec_joined_spatialgrain,
  CRP %>% select(seqnum, year, CRP_area, CRPGrass_area),
  by = c("seqnum", "year")
)

In [ ]:
# Join detailed CRP landscape metrics
CRP2 <- read.csv(file.choose(), header = TRUE)[, c(1:9, 11:14, 16:19)]
CRP2 <- CRP2 %>%
  separate(cell_year, into = c("seqnum", "year"), sep = "_") %>%
  mutate(seqnum = as.numeric(seqnum), year = as.numeric(year))

spec_joined_spatialgrain <- left_join(
  spec_joined_spatialgrain,
  CRP2 %>% select(
    seqnum, year,
    Hab_PercentGrassland, Hab_PercentWetland,
    obj_PercentGrass, obj_PercentWetland, obj_PercentWildlife,
    brd_PercentAttract, brd_PercentNeutral, brd_PercentAvoid,
    CRP_largest_patch_index, CRP_patch_density, CRP_edge_density, CRP_contagion,
    CRPGrass_largest_patch_index, CRPGrass_patch_density,
    CRPGrass_edge_density, CRPGrass_contagion
  ),
  by = c("seqnum", "year")
)

# Classify experimental vs. control
spec_joined_spatialgrain$data_type <- NA_character_
spec_joined_spatialgrain$data_type[
  spec_joined_spatialgrain$CRP_area_ha == 0 & spec_joined_spatialgrain$WindCount == 0
] <- "control"
spec_joined_spatialgrain$data_type[
  spec_joined_spatialgrain$CRP_area_ha > 0 | spec_joined_spatialgrain$WindCount > 0
] <- "exp"

In [ ]:
# Subsample control to reduce imbalance in developed land cover
exp     <- spec_joined_spatialgrain %>% filter(data_type == "exp")
control <- spec_joined_spatialgrain %>% filter(data_type == "control")

control_greaterQ <- control %>% filter(Developed > quantile(exp$Developed, 0.75))
control_lessQ    <- control %>% filter(Developed <= quantile(exp$Developed, 0.75))

GQcount            <- floor(0.25 * nrow(control_greaterQ))
control_greaterQ_ss <- control_greaterQ %>% sample_n(size = GQcount)
control_ss         <- bind_rows(control_lessQ, control_greaterQ_ss)

# Sanity checks
summary(exp$Developed)
summary(control$Developed)
summary(control_ss$Developed)

# Recombine and inspect prevalence
spec_joined_spatialgrain_ss <- bind_rows(control_ss, exp)
spec_joined_spatialgrain_ss %>%
  count(species_observed) %>%
  mutate(percent = n / sum(n))

spec_final <- spec_joined_spatialgrain_ss

# Group counts
spec_final %>% count(CRP_area_ha > 0 & WindCount > 0)  %>% mutate(percent = n / sum(n))  # CRP & turbine
spec_final %>% count(CRP_area_ha > 0 & WindCount == 0) %>% mutate(percent = n / sum(n))  # CRP only
spec_final %>% count(CRP_area_ha == 0 & WindCount > 0) %>% mutate(percent = n / sum(n))  # turbine only
spec_final %>% count(CRP_area_ha == 0 & WindCount == 0) %>% mutate(percent = n / sum(n)) # neither

---
## Part 4: Wind Variable Categorization

Continuous wind turbine attributes (count, age, hub height, rotor swept area, capacity) are binned into ordered categorical variables to allow flexible nonlinear or step-wise effects in GAMs.

In [ ]:
# Presence/absence
spec_joined_spatialgrain$WindPA <- as.factor(
  ifelse(spec_joined_spatialgrain$WindCount > 0, "Y", "N")
)

# Age bins
spec_joined_spatialgrain$WindAge_cat <- ifelse(spec_joined_spatialgrain$WindPA == "N", "None",
  ifelse(spec_joined_spatialgrain$WindAge <= 2, "0-2",
  ifelse(spec_joined_spatialgrain$WindAge <= 4, "3-4",
  ifelse(spec_joined_spatialgrain$WindAge <= 6, "5-6",
  ifelse(spec_joined_spatialgrain$WindAge <= 8, "7-8", "9+")))))
spec_joined_spatialgrain$WindAge_cat <- factor(
  spec_joined_spatialgrain$WindAge_cat, levels = c("None","0-2","3-4","5-6","7-8","9+")
)

# Hub height bins
spec_joined_spatialgrain$WindHeight_cat <- ifelse(spec_joined_spatialgrain$WindPA == "N", "None",
  ifelse(spec_joined_spatialgrain$WindHeight <= 20,  "1-20",
  ifelse(spec_joined_spatialgrain$WindHeight <= 40,  "21-40",
  ifelse(spec_joined_spatialgrain$WindHeight <= 60,  "41-60",
  ifelse(spec_joined_spatialgrain$WindHeight <= 80,  "61-80", "81-100+")))))
spec_joined_spatialgrain$WindHeight_cat <- factor(
  spec_joined_spatialgrain$WindHeight_cat,
  levels = c("None","1-20","21-40","41-60","61-80","81-100+")
)

# Rotor swept area bins (m²)
spec_joined_spatialgrain$WindRSA_cat <- ifelse(spec_joined_spatialgrain$WindPA == "N", "None",
  ifelse(spec_joined_spatialgrain$WindRSA <= 5000,  "0-5000",
  ifelse(spec_joined_spatialgrain$WindRSA <= 10000, "5001-10000",
  ifelse(spec_joined_spatialgrain$WindRSA <= 15000, "10001-15000", "15000+"))))
spec_joined_spatialgrain$WindRSA_cat <- factor(
  spec_joined_spatialgrain$WindRSA_cat,
  levels = c("None","0-5000","5001-10000","10001-15000","15000+")
)

# Capacity bins (kW)
spec_joined_spatialgrain$WindCap_cat <- ifelse(spec_joined_spatialgrain$WindPA == "N", "None",
  ifelse(spec_joined_spatialgrain$WindCap <= 1000, "0-1000",
  ifelse(spec_joined_spatialgrain$WindCap <= 2000, "1001-2000",
  ifelse(spec_joined_spatialgrain$WindCap <= 3000, "2001-3000",
  ifelse(spec_joined_spatialgrain$WindCap <= 4000, "3001-4000", "4001+")))))
spec_joined_spatialgrain$WindCap_cat <- factor(
  spec_joined_spatialgrain$WindCap_cat,
  levels = c("None","0-1000","1001-2000","2001-3000","3001-4000","4001+")
)

# Drop rows missing key wind attributes
spec_filtered_spatialgrain <- spec_joined_spatialgrain %>%
  drop_na(WindHeight_cat, WindCap_cat, WindRSA_cat)

---
## Part 5: Correlation Matrix

A Pearson correlation matrix is computed across all candidate covariates to identify multicollinearity prior to model construction. Results are saved for inspection.

In [ ]:
cortest <- spec_filtered_spatialgrain %>%
  select(
    WindCount,
    Developed, Cropland, GrassShrub, TreeCover, Water, Wetland, Barren, IceSnow,
    meanStartTime, meanSampDur, meanDIST, meanNumObs,
    CRP_area, CRP_largest_patch_index, CRP_patch_density, CRP_edge_density, CRP_contagion,
    CRPGrass_area, CRPGrass_largest_patch_index, CRPGrass_patch_density,
    CRPGrass_edge_density, CRPGrass_contagion,
    Hab_PercentGrassland, Hab_PercentWetland,
    obj_PercentGrass, obj_PercentWetland, obj_PercentWildlife,
    brd_PercentAttract, brd_PercentNeutral, brd_PercentAvoid
  )

corr_matrix <- cor(cortest, use = "complete.obs")

# Reorder columns for visual grouping
corr_matrix <- corr_matrix[, c(
  "Developed","Cropland","GrassShrub","TreeCover","Water","Wetland","Barren","IceSnow",
  "meanStartTime","meanSampDur","meanDIST","meanNumObs","WindCount",
  "CRP_area","CRP_largest_patch_index","CRP_patch_density","CRP_edge_density","CRP_contagion",
  "CRPGrass_area","CRPGrass_largest_patch_index","CRPGrass_patch_density",
  "CRPGrass_edge_density","CRPGrass_contagion",
  "Hab_PercentGrassland","Hab_PercentWetland",
  "obj_PercentGrass","obj_PercentWetland","obj_PercentWildlife",
  "brd_PercentAttract","brd_PercentNeutral","brd_PercentAvoid"
)]

write.csv(corr_matrix, "spec_corr_matrix.csv", row.names = TRUE)

# Visualize
corrplot(corr_matrix, method = "color", tl.cex = 0.6)

---
## Part 6: Standardization and Train/Test Split

All continuous covariates are z-score standardized. Scaling parameters (mean, SD) are saved to allow back-transformation for figure generation. Data are split 80/20 into training and test sets.

In [ ]:
scaled_cols <- spec_filtered_spatialgrain %>%
  select(
    meanStartTime, meanSampDur, meanDIST, meanNumObs,
    Developed, Cropland, GrassShrub, TreeCover, Water, Wetland, Barren, IceSnow,
    WindCount,
    Hab_PercentGrassland, Hab_PercentWetland,
    CRP_area, CRP_largest_patch_index, CRP_patch_density, CRP_edge_density, CRP_contagion,
    CRPGrass_area, CRPGrass_largest_patch_index, CRPGrass_patch_density,
    CRPGrass_edge_density, CRPGrass_contagion,
    obj_PercentGrass, obj_PercentWetland, obj_PercentWildlife,
    brd_PercentAttract, brd_PercentNeutral, brd_PercentAvoid,
    Lat, Long
  )

# Save scaling parameters for back-transformation
scaling_params <- scaled_cols %>%
  summarise(across(where(is.numeric),
    list(mean = ~ mean(., na.rm = TRUE), sd = ~ sd(., na.rm = TRUE)))) %>%
  pivot_longer(everything(), names_to = c("variable", "statistic"), names_sep = "_",
               values_to = "value") %>%
  pivot_wider(names_from = "statistic", values_from = "value")

scaled_data <- scaled_cols %>% mutate(across(where(is.numeric), scale))

spec_scaled_spatialgrain <- cbind(
  meanCount       = spec_filtered_spatialgrain$meanCount,
  meanCount_round = spec_filtered_spatialgrain$meanCount_round,
  WindAge         = spec_filtered_spatialgrain$WindAge_cat,
  WindHeight      = spec_filtered_spatialgrain$WindHeight_cat,
  WindCap         = spec_filtered_spatialgrain$WindCap_cat,
  WindRSA         = spec_filtered_spatialgrain$WindRSA_cat,
  WindPA          = spec_filtered_spatialgrain$WindPA,
  scaled_data
)

In [ ]:
# Select model inputs and perform 80/20 random split
spec_split_spatialgrain <- spec_scaled_spatialgrain %>%
  select(
    meanCount, meanCount_round,
    meanStartTime, meanSampDur, meanDIST, meanNumObs,
    Developed, Cropland, GrassShrub, TreeCover, Water, Wetland, Barren, IceSnow,
    WindCount, WindHeight, WindCap, WindAge, WindPA, WindRSA,
    Hab_PercentGrassland, Hab_PercentWetland,
    CRP_area, CRP_largest_patch_index, CRP_patch_density, CRP_edge_density, CRP_contagion,
    CRPGrass_area, CRPGrass_largest_patch_index, CRPGrass_patch_density,
    CRPGrass_edge_density, CRPGrass_contagion,
    obj_PercentGrass, obj_PercentWetland, obj_PercentWildlife,
    brd_PercentAttract, brd_PercentNeutral, brd_PercentAvoid,
    Lat, Long
  ) %>%
  split(if_else(runif(nrow(.)) <= 0.8, "train", "test"))

map_int(spec_split_spatialgrain, nrow)

---
## Part 7: Distribution Family Evaluation Using Null Models

Null GAMs (intercept only) are fit under Poisson, negative binomial, zero-inflated Poisson, and zero-inflated negative binomial families. AIC/BIC are used to select the most appropriate count distribution for the response variable.

In [ ]:
# GAM smoothing parameters
k        <- 5   # knots for most smooths
k_time   <- 7   # knots for cyclic time-of-day smooth
time_knots <- list(meanStartTime = seq(0, 24, length.out = k_time))

Null <- meanCount_round ~ 1

# Negative binomial
start.time <- Sys.time()
gam_nb   <- gam(Null, data = spec_split_spatialgrain$train, family = "nb", knots = time_knots)
cat("NB:", round(Sys.time() - start.time, 2), "sec\n")

# Poisson
start.time <- Sys.time()
gam_pois <- gam(Null, data = spec_split_spatialgrain$train, family = "poisson", knots = time_knots)
cat("Poisson:", round(Sys.time() - start.time, 2), "sec\n")

# Zero-inflated NB
start.time <- Sys.time()
gam_zinb <- zinbgam(Null, pi.formula = ~1, data = spec_split_spatialgrain$train, knots = time_knots)
cat("ZINB:", round(Sys.time() - start.time, 2), "sec\n")

# Zero-inflated Poisson
start.time <- Sys.time()
gam_zip  <- gam(Null, data = spec_split_spatialgrain$train, family = "ziP", knots = time_knots)
cat("ZIP:", round(Sys.time() - start.time, 2), "sec\n")

# Quasi-Poisson (for overdispersion check; no AIC)
gam_qp <- gam(Null, data = spec_split_spatialgrain$train, family = "quasipoisson", knots = time_knots)

# Compare
AIC(gam_nb, gam_pois, gam_zip)
BIC(gam_nb, gam_pois, gam_zip)

# Manual BIC for ZINB (not directly available)
AIC_zinb <- gam_zinb$aic
BIC_zinb <- AIC_zinb + log(nrow(spec_split_spatialgrain$train)) *
            (length(coef(gam_zinb)) - length(fitted(gam_zinb)) + 1)
cat("ZINB AIC:", AIC_zinb, "\nZINB BIC:", BIC_zinb, "\n")

# Quasi-Poisson summary
sum.gam <- summary(gam_qp)
sum.gam$p.pv  # parametric p-values
sum.gam$s.pv  # smooth term p-values
sum(residuals(gam_qp, type = "response"))

---
## Part 8: Univariate Variable Screening (Effort and Land Cover)

Each candidate covariate is added individually to a base spatial model to evaluate its marginal contribution via effective degrees of freedom (edf) and AICc. This screens for variables worth retaining in full models.

In [ ]:
# --- Effort variable screening ---
base            <- meanCount ~ s(Lat, k=5) + s(Long, k=5)
effort_SampDur  <- update(base, . ~ . + s(meanSampDur, k=5))
effort_NumObs   <- update(base, . ~ . + s(meanNumObs, k=5))
effort_Dist     <- update(base, . ~ . + s(meanDIST, k=5))
effort_StartTime <- update(base, . ~ . + s(meanStartTime, bs="cc", k=7))
effort_Global   <- meanCount ~ s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
                               s(meanStartTime,bs="cc",k=7) + s(Lat,k=5) + s(Long,k=5)

m_effort_base      <- gam(base,             data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_effort_SampDur   <- gam(effort_SampDur,   data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_effort_NumObs    <- gam(effort_NumObs,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_effort_Dist      <- gam(effort_Dist,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_effort_StartTime <- gam(effort_StartTime, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_effort_global    <- gam(effort_Global,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)

MuMIn::AICc(m_effort_base, m_effort_SampDur, m_effort_NumObs,
            m_effort_Dist, m_effort_StartTime, m_effort_global)

In [ ]:
# --- Land cover variable screening ---
lc_base      <- meanCount ~ s(Lat,k=5) + s(Long,k=5)
lc_developed <- update(lc_base, . ~ . + s(Developed,k=5))
lc_cropland  <- update(lc_base, . ~ . + s(Cropland,k=5))
lc_grasshrub <- update(lc_base, . ~ . + s(GrassShrub,k=5))
lc_treecover <- update(lc_base, . ~ . + s(TreeCover,k=5))
lc_water     <- update(lc_base, . ~ . + s(Water,k=5))
lc_wetland   <- update(lc_base, . ~ . + s(Wetland,k=5))
lc_barren    <- update(lc_base, . ~ . + s(Barren,k=5))
lc_icesnow   <- update(lc_base, . ~ . + s(IceSnow,k=5))

m_lc_base      <- gam(lc_base,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_lc_developed <- gam(lc_developed, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_lc_cropland  <- gam(lc_cropland,  data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_lc_grasshrub <- gam(lc_grasshrub, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_lc_treecover <- gam(lc_treecover, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_lc_water     <- gam(lc_water,     data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_lc_wetland   <- gam(lc_wetland,   data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_lc_barren    <- gam(lc_barren,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_lc_icesnow   <- gam(lc_icesnow,   data=spec_split_spatialgrain$train, family="nb", knots=time_knots)

MuMIn::AICc(m_lc_base, m_lc_developed, m_lc_cropland, m_lc_grasshrub,
            m_lc_treecover, m_lc_water, m_lc_wetland, m_lc_barren, m_lc_icesnow)

---
## Part 9: Wind Variable Screening (Simple Spatial Models)

Each wind metric is added individually to a spatial-only base model to compare AICc and identify the best-supported representation of wind turbine presence and characteristics.

In [ ]:
Wind_base           <- meanCount ~ s(Lat,k=5) + s(Long,k=5)
Wind_windcount_simple <- update(Wind_base, . ~ . + s(WindCount,k=5))
Wind_windPA_simple    <- update(Wind_base, . ~ . + factor(WindPA))
Wind_windrsa_simple   <- update(Wind_base, . ~ . + factor(WindRSA))
Wind_windheight_simple <- update(Wind_base, . ~ . + factor(WindHeight))
Wind_windage_simple   <- update(Wind_base, . ~ . + factor(WindAge))
Wind_windcap_simple   <- update(Wind_base, . ~ . + factor(WindCap))

m_wind_count_simple  <- gam(Wind_windcount_simple,  data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_cap_simple    <- gam(Wind_windcap_simple,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_Wind_windPA_simple <- gam(Wind_windPA_simple,     data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_rsa_simple    <- gam(Wind_windrsa_simple,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_height_simple <- gam(Wind_windheight_simple, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_age_simple    <- gam(Wind_windage_simple,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)

MuMIn::AICc(m_wind_count_simple, m_wind_cap_simple, m_Wind_windPA_simple,
            m_wind_rsa_simple, m_wind_height_simple, m_wind_age_simple, m_lc_base)

---
## Part 10: CRP Variable Screening (Simple Spatial Models)

Each CRP metric (area, patch geometry, habitat objectives, bird community attractiveness) is screened individually against a spatial base model using AICc.

In [ ]:
CRP_Hab_PercentGrassland_simple    <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(Hab_PercentGrassland,k=5)
CRP_Hab_PercentWetland_simple       <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(Hab_PercentWetland,k=5)
CRP_CRP_area_simple                 <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRP_area,k=5)
CRP_CRP_largest_patch_index_simple  <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRP_largest_patch_index,k=5)
CRP_CRP_patch_density_simple        <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRP_patch_density,k=5)
CRP_CRP_edge_density_simple         <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRP_edge_density,k=5)
CRP_CRP_contagion_simple            <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRP_contagion,k=5)
CRP_CRPGrass_area_simple            <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRPGrass_area,k=5)
CRP_CRPGrass_largest_patch_index_simple <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRPGrass_largest_patch_index,k=5)
CRP_CRPGrass_patch_density_simple   <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRPGrass_patch_density,k=5)
CRP_CRPGrass_edge_density_simple    <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRPGrass_edge_density,k=5)
CRP_CRPGrass_contagion_simple       <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(CRPGrass_contagion,k=5)
CRP_obj_PercentGrass_simple         <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(obj_PercentGrass,k=5)
CRP_obj_PercentWetland_simple       <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(obj_PercentWetland,k=5)
CRP_obj_PercentWildlife_simple      <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(obj_PercentWildlife,k=5)
CRP_brd_PercentAttract_simple       <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(brd_PercentAttract,k=5)
CRP_brd_PercentNeutral_simple       <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(brd_PercentNeutral,k=5)
CRP_brd_PercentAvoid_simple         <- meanCount ~ s(Lat,k=5) + s(Long,k=5) + s(brd_PercentAvoid,k=5)
CRP_base_simple                     <- meanCount ~ s(Lat,k=5) + s(Long,k=5)

start.time <- Sys.time()
m_CRP_Hab_PercentGrassland_simple    <- gam(CRP_Hab_PercentGrassland_simple,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_Hab_PercentGrassland_simple:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_Hab_PercentGrassland_simple)

start.time <- Sys.time()
m_CRP_Hab_PercentWetland_simple      <- gam(CRP_Hab_PercentWetland_simple,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_Hab_PercentWetland_simple:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_Hab_PercentWetland_simple)

start.time <- Sys.time()
m_CRP_CRP_area_simple                <- gam(CRP_CRP_area_simple,                data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_area_simple:",                round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_area_simple)

start.time <- Sys.time()
m_CRP_CRP_largest_patch_index_simple <- gam(CRP_CRP_largest_patch_index_simple, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_largest_patch_index_simple:", round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_largest_patch_index_simple)

start.time <- Sys.time()
m_CRP_CRP_patch_density_simple       <- gam(CRP_CRP_patch_density_simple,       data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_patch_density_simple:",       round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_patch_density_simple)

start.time <- Sys.time()
m_CRP_CRP_edge_density_simple        <- gam(CRP_CRP_edge_density_simple,        data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_edge_density_simple:",        round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_edge_density_simple)

start.time <- Sys.time()
m_CRP_CRP_contagion_simple           <- gam(CRP_CRP_contagion_simple,           data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_contagion_simple:",           round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_contagion_simple)

start.time <- Sys.time()
m_CRP_CRPGrass_area_simple           <- gam(CRP_CRPGrass_area_simple,           data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_area_simple:",           round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_area_simple)

start.time <- Sys.time()
m_CRP_CRPGrass_largest_patch_index_simple <- gam(CRP_CRPGrass_largest_patch_index_simple, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_largest_patch_index_simple:", round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_largest_patch_index_simple)

start.time <- Sys.time()
m_CRP_CRPGrass_patch_density_simple  <- gam(CRP_CRPGrass_patch_density_simple,  data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_patch_density_simple:",  round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_patch_density_simple)

start.time <- Sys.time()
m_CRP_CRPGrass_edge_density_simple   <- gam(CRP_CRPGrass_edge_density_simple,   data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_edge_density_simple:",   round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_edge_density_simple)

start.time <- Sys.time()
m_CRP_CRPGrass_contagion_simple      <- gam(CRP_CRPGrass_contagion_simple,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_contagion_simple:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_contagion_simple)

start.time <- Sys.time()
m_CRP_obj_PercentGrass_simple        <- gam(CRP_obj_PercentGrass_simple,        data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_obj_PercentGrass_simple:",        round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_obj_PercentGrass_simple)

start.time <- Sys.time()
m_CRP_obj_PercentWetland_simple      <- gam(CRP_obj_PercentWetland_simple,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_obj_PercentWetland_simple:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_obj_PercentWetland_simple)

start.time <- Sys.time()
m_CRP_obj_PercentWildlife_simple     <- gam(CRP_obj_PercentWildlife_simple,     data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_obj_PercentWildlife_simple:",     round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_obj_PercentWildlife_simple)

start.time <- Sys.time()
m_CRP_brd_PercentAttract_simple      <- gam(CRP_brd_PercentAttract_simple,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_brd_PercentAttract_simple:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_brd_PercentAttract_simple)

start.time <- Sys.time()
m_CRP_brd_PercentNeutral_simple      <- gam(CRP_brd_PercentNeutral_simple,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_brd_PercentNeutral_simple:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_brd_PercentNeutral_simple)

start.time <- Sys.time()
m_CRP_brd_PercentAvoid_simple        <- gam(CRP_brd_PercentAvoid_simple,        data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_brd_PercentAvoid_simple:",        round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_brd_PercentAvoid_simple)

start.time <- Sys.time()
m_CRP_base_simple                    <- gam(CRP_base_simple,                    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_base_simple:",                    round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_base_simple)

MuMIn::AICc(
  m_CRP_Hab_PercentGrassland_simple, m_CRP_Hab_PercentWetland_simple,
  m_CRP_CRP_area_simple, m_CRP_CRP_largest_patch_index_simple,
  m_CRP_CRP_patch_density_simple, m_CRP_CRP_edge_density_simple, m_CRP_CRP_contagion_simple,
  m_CRP_CRPGrass_area_simple, m_CRP_CRPGrass_largest_patch_index_simple,
  m_CRP_CRPGrass_patch_density_simple, m_CRP_CRPGrass_edge_density_simple, m_CRP_CRPGrass_contagion_simple,
  m_CRP_obj_PercentGrass_simple, m_CRP_obj_PercentWetland_simple, m_CRP_obj_PercentWildlife_simple,
  m_CRP_brd_PercentAttract_simple, m_CRP_brd_PercentNeutral_simple, m_CRP_brd_PercentAvoid_simple,
  m_CRP_base_simple
)

---
## Part 11: Wind-Only Full Models (with Effort and Land Cover)

Full candidate wind models are fit including all retained effort and land cover covariates. Replace `wind_variable` with the best-supported wind metric identified in Part 9.

In [ ]:
# Replace 'wind_variable' with the selected wind covariate name (e.g., WindCount, WindCap)

base_full <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) +
  s(Lat,k=5) + s(Long,k=5)

Wind_windcount <- update(base_full, . ~ . + s(WindCount, k=5))
Wind_windcap   <- update(base_full, . ~ . + factor(WindCap))
Wind_windPA    <- update(base_full, . ~ . + factor(WindPA))
Wind_windrsa   <- update(base_full, . ~ . + factor(WindRSA))
Wind_windheight <- update(base_full, . ~ . + factor(WindHeight))
Wind_windage   <- update(base_full, . ~ . + factor(WindAge))

m_wind_count  <- gam(Wind_windcount,  data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_cap    <- gam(Wind_windcap,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_Wind_windPA <- gam(Wind_windPA,     data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_rsa    <- gam(Wind_windrsa,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_height <- gam(Wind_windheight, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_age    <- gam(Wind_windage,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_base   <- gam(base_full,       data=spec_split_spatialgrain$train, family="nb", knots=time_knots)

MuMIn::AICc(m_wind_count, m_wind_cap, m_Wind_windPA,
            m_wind_rsa, m_wind_height, m_wind_age, m_wind_base)

---
## Part 12: CRP-Only Full Models (with Effort and Land Cover)

Full candidate CRP models are fit including all retained effort and land cover covariates. Replace `crp_variable` with the best-supported CRP metric identified in Part 10.

In [ ]:
# Replace 'crp_variable' with the selected CRP covariate name (e.g., CRP_area, Hab_PercentGrassland)

CRP_Hab_PercentGrassland    <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(Hab_PercentGrassland,k=5)
CRP_Hab_PercentWetland      <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(Hab_PercentWetland,k=5)
CRP_CRP_area                <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRP_area,k=5)
CRP_CRP_largest_patch_index <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRP_largest_patch_index,k=5)
CRP_CRP_patch_density       <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRP_patch_density,k=5)
CRP_CRP_edge_density        <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRP_edge_density,k=5)
CRP_CRP_contagion           <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRP_contagion,k=5)
CRP_CRPGrass_area           <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRPGrass_area,k=5)
CRP_CRPGrass_largest_patch_index <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRPGrass_largest_patch_index,k=5)
CRP_CRPGrass_patch_density  <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRPGrass_patch_density,k=5)
CRP_CRPGrass_edge_density   <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRPGrass_edge_density,k=5)
CRP_CRPGrass_contagion      <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(CRPGrass_contagion,k=5)
CRP_obj_PercentGrass        <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(obj_PercentGrass,k=5)
CRP_obj_PercentWetland      <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(obj_PercentWetland,k=5)
CRP_obj_PercentWildlife     <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(obj_PercentWildlife,k=5)
CRP_brd_PercentAttract      <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(brd_PercentAttract,k=5)
CRP_brd_PercentNeutral      <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(brd_PercentNeutral,k=5)
CRP_brd_PercentAvoid        <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5) +
  s(brd_PercentAvoid,k=5)
CRP_base                    <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) + s(Lat,k=5) + s(Long,k=5)

start.time <- Sys.time()
m_CRP_Hab_PercentGrassland    <- gam(CRP_Hab_PercentGrassland,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_Hab_PercentGrassland:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_Hab_PercentGrassland)

start.time <- Sys.time()
m_CRP_Hab_PercentWetland      <- gam(CRP_Hab_PercentWetland,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_Hab_PercentWetland:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_Hab_PercentWetland)

start.time <- Sys.time()
m_CRP_CRP_area                <- gam(CRP_CRP_area,                data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_area:",                round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_area)

start.time <- Sys.time()
m_CRP_CRP_largest_patch_index <- gam(CRP_CRP_largest_patch_index, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_largest_patch_index:", round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_largest_patch_index)

start.time <- Sys.time()
m_CRP_CRP_patch_density       <- gam(CRP_CRP_patch_density,       data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_patch_density:",       round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_patch_density)

start.time <- Sys.time()
m_CRP_CRP_edge_density        <- gam(CRP_CRP_edge_density,        data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_edge_density:",        round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_edge_density)

start.time <- Sys.time()
m_CRP_CRP_contagion           <- gam(CRP_CRP_contagion,           data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRP_contagion:",           round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRP_contagion)

start.time <- Sys.time()
m_CRP_CRPGrass_area           <- gam(CRP_CRPGrass_area,           data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_area:",           round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_area)

start.time <- Sys.time()
m_CRP_CRPGrass_largest_patch_index <- gam(CRP_CRPGrass_largest_patch_index, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_largest_patch_index:", round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_largest_patch_index)

start.time <- Sys.time()
m_CRP_CRPGrass_patch_density  <- gam(CRP_CRPGrass_patch_density,  data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_patch_density:",  round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_patch_density)

start.time <- Sys.time()
m_CRP_CRPGrass_edge_density   <- gam(CRP_CRPGrass_edge_density,   data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_edge_density:",   round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_edge_density)

start.time <- Sys.time()
m_CRP_CRPGrass_contagion      <- gam(CRP_CRPGrass_contagion,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_CRPGrass_contagion:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_CRPGrass_contagion)

start.time <- Sys.time()
m_CRP_obj_PercentGrass        <- gam(CRP_obj_PercentGrass,        data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_obj_PercentGrass:",        round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_obj_PercentGrass)

start.time <- Sys.time()
m_CRP_obj_PercentWetland      <- gam(CRP_obj_PercentWetland,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_obj_PercentWetland:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_obj_PercentWetland)

start.time <- Sys.time()
m_CRP_obj_PercentWildlife     <- gam(CRP_obj_PercentWildlife,     data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_obj_PercentWildlife:",     round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_obj_PercentWildlife)

start.time <- Sys.time()
m_CRP_brd_PercentAttract      <- gam(CRP_brd_PercentAttract,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_brd_PercentAttract:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_brd_PercentAttract)

start.time <- Sys.time()
m_CRP_brd_PercentNeutral      <- gam(CRP_brd_PercentNeutral,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_brd_PercentNeutral:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_brd_PercentNeutral)

start.time <- Sys.time()
m_CRP_brd_PercentAvoid        <- gam(CRP_brd_PercentAvoid,        data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_brd_PercentAvoid:",        round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_brd_PercentAvoid)

start.time <- Sys.time()
m_CRP_base                    <- gam(CRP_base,                    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_CRP_base:",                    round(Sys.time()-start.time,2), "sec\n"); summary(m_CRP_base)

MuMIn::AICc(
  m_CRP_Hab_PercentGrassland, m_CRP_Hab_PercentWetland,
  m_CRP_CRP_area, m_CRP_CRP_largest_patch_index,
  m_CRP_CRP_patch_density, m_CRP_CRP_edge_density, m_CRP_CRP_contagion,
  m_CRP_CRPGrass_area, m_CRP_CRPGrass_largest_patch_index,
  m_CRP_CRPGrass_patch_density, m_CRP_CRPGrass_edge_density, m_CRP_CRPGrass_contagion,
  m_CRP_obj_PercentGrass, m_CRP_obj_PercentWetland, m_CRP_obj_PercentWildlife,
  m_CRP_brd_PercentAttract, m_CRP_brd_PercentNeutral, m_CRP_brd_PercentAvoid,
  m_CRP_base
)

---
## Part 13: Combined Wind + CRP Models

Seven candidate model structures are compared, ranging from additive to interaction forms. Replace `wind_variable` and `crp_variable` with the best-supported metrics from Parts 11–12.

In [ ]:
# IMPORTANT: Replace 'wind_variable' and 'crp_variable' with actual column names before running.
# Example: wind_variable -> WindCount (continuous) or WindPA (categorical)
#          crp_variable  -> CRP_area

wind_categorical_model <- update(base_full, . ~ . + factor(wind_variable))
wind_nonlinear_model   <- update(base_full, . ~ . + s(wind_variable, k=5))
crp_model              <- update(base_full, . ~ . + s(crp_variable, k=5))

interaction_model <- update(base_full, . ~ . +
  factor(wind_variable) + s(crp_variable, k=5) + factor(wind_variable):crp_variable)

interaction_model_smooths <- update(base_full, . ~ . + te(wind_variable, crp_variable, k=5))

additive_model        <- update(base_full, . ~ . + factor(wind_variable) + s(crp_variable, k=5))
additive_model_smooths <- update(base_full, . ~ . + s(wind_variable, k=5) + s(crp_variable, k=5))

m_wind_categorical    <- gam(wind_categorical_model,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_wind_nonlinear      <- gam(wind_nonlinear_model,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_crp                 <- gam(crp_model,                 data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_interaction         <- gam(interaction_model,         data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_interaction_smooths <- gam(interaction_model_smooths, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_additive            <- gam(additive_model,            data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
m_additive_smooths    <- gam(additive_model_smooths,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)

MuMIn::AICc(m_wind_categorical, m_wind_nonlinear, m_crp,
            m_interaction, m_interaction_smooths, m_additive, m_additive_smooths)

---
## Part 14: Top Model Diagnostics

Variance components and ANOVA tables are inspected for the top candidate models. Default GAM diagnostic plots are also produced.

In [ ]:
# Replace model objects as appropriate based on AICc results

# Variance components (rescaled)
gam.vcomp(m_wind_categorical,    rescale = TRUE)
gam.vcomp(m_crp,                 rescale = TRUE)
gam.vcomp(m_interaction,         rescale = TRUE)
gam.vcomp(m_additive,            rescale = TRUE)

# ANOVA tables
anova(m_wind_categorical)
anova(m_crp)
anova(m_interaction)
anova(m_additive)

# Default diagnostic plots
par(mfrow = c(2, 2))
plot(m_interaction)  # replace with top model

---
## Part 15: Figure Generation

Predicted abundance from each model class is visualized against the focal predictors. Continuous variables are back-transformed using saved scaling parameters for interpretable axes.

In [ ]:
# Back-transform standardized variables for plot axes
scaling_params_sub <- scaling_params %>%
  filter(variable %in% c("crp_variable", "wind_variable"))

spec_split_spatialgrain$test$crp_variable_unscaled <-
  spec_split_spatialgrain$test$crp_variable *
  scaling_params_sub$sd[scaling_params_sub$variable == "crp_variable"] +
  scaling_params_sub$mean[scaling_params_sub$variable == "crp_variable"]

spec_split_spatialgrain$test$wind_variable_unscaled <-
  spec_split_spatialgrain$test$wind_variable *
  scaling_params_sub$sd[scaling_params_sub$variable == "wind_variable"] +
  scaling_params_sub$mean[scaling_params_sub$variable == "wind_variable"]

In [ ]:
# Figure 1: Continuous wind variable only
pred_wind_nl <- predict.gam(m_wind_nonlinear, newdata=spec_split_spatialgrain$test,
                            type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_wind_nl <- spec_split_spatialgrain$test
pred_df_wind_nl$predicted <- pred_wind_nl$fit

ggplot(pred_df_wind_nl, aes(x=wind_variable_unscaled, y=predicted)) +
  geom_smooth() +
  labs(x="Wind Variable", y="Predicted Abundance") +
  theme_minimal()

In [ ]:
# Figure 2: Categorical wind variable only
pred_wind_cat <- predict.gam(m_wind_categorical, newdata=spec_split_spatialgrain$test,
                             type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_wind_cat <- spec_split_spatialgrain$test
pred_df_wind_cat$predicted <- pred_wind_cat$fit

ggplot(pred_df_wind_cat, aes(x=wind_variable, y=predicted)) +
  geom_boxplot() +
  labs(x="Wind Variable", y="Predicted Abundance") +
  theme_minimal()

In [ ]:
# Figure 3: CRP variable only
pred_crp <- predict.gam(m_crp, newdata=spec_split_spatialgrain$test,
                        type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_crp <- spec_split_spatialgrain$test
pred_df_crp$predicted <- pred_crp$fit

ggplot(pred_df_crp, aes(x=crp_variable_unscaled, y=predicted)) +
  geom_smooth() +
  labs(x="CRP Variable", y="Predicted Abundance") +
  theme_minimal()

In [ ]:
# Figure 4: Additive model — categorical wind + continuous CRP
pred_add_cat <- predict.gam(m_additive, newdata=spec_split_spatialgrain$test,
                            type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_add_cat <- spec_split_spatialgrain$test
pred_df_add_cat$predicted <- pred_add_cat$fit

ggplot(pred_df_add_cat, aes(x=crp_variable_unscaled, y=predicted, color=wind_variable)) +
  geom_smooth() +
  labs(x="CRP Variable", y="Predicted Abundance", color="Wind Variable") +
  theme_minimal()

In [ ]:
# Figure 5: Additive model — continuous wind + continuous CRP
pred_add_nl <- predict.gam(m_additive_smooths, newdata=spec_split_spatialgrain$test,
                           type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_add_nl <- spec_split_spatialgrain$test
pred_df_add_nl$predicted <- pred_add_nl$fit

ggplot(pred_df_add_nl, aes(x=crp_variable_unscaled, y=predicted, color=wind_variable)) +
  geom_point(alpha=0.3) +
  geom_smooth() +
  labs(x="CRP Variable", y="Predicted Abundance", color="Wind Variable") +
  theme_minimal()

In [ ]:
# Figure 6: Interaction model — categorical wind × continuous CRP
pred_inter_cat <- predict.gam(m_interaction, newdata=spec_split_spatialgrain$test,
                              type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_inter_cat <- spec_split_spatialgrain$test
pred_df_inter_cat$predicted <- pred_inter_cat$fit

ggplot(pred_df_inter_cat, aes(x=crp_variable_unscaled, y=predicted, color=wind_variable)) +
  geom_smooth() +
  labs(x="CRP Variable", y="Predicted Abundance", color="Wind Variable") +
  theme_minimal()

In [ ]:
# Figure 7: Interaction model — continuous wind × continuous CRP (2D heatmap)
pred_inter_smooth <- predict.gam(m_interaction_smooths, newdata=spec_split_spatialgrain$test,
                                 type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_inter_smooth <- spec_split_spatialgrain$test
pred_df_inter_smooth$predicted <- pred_inter_smooth$fit

ggplot(pred_df_inter_smooth,
       aes(x=wind_variable_unscaled, y=crp_variable_unscaled, fill=predicted)) +
  geom_tile() +
  labs(x="Wind Variable", y="CRP Variable", fill="Predicted Abundance") +
  theme_minimal()